# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [ ]:
import Bio.SeqIO
import Bio.Seq
import re


# SeqIO.parse to iterate over the file. Returns reads one at a time and list() to pull them into one item
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

upstream = "AGGCGGCCGC"

def HANA_classifier (seqreads): 

    match_na_count = 0
    match_ha_count = 0

# iterate over each sequence read in list of sequence reads 
    for seqread in seqreads:
    # extract sequence data from seqread object and convert it into string and upper case. 
        seq = seqread.seq
        seq_string = str(seq)
        seq_upper = seq_string.upper()

    #reverse complement extracted uppercase sequence (is this now the actual sequence)
        seq_original_string = str(Bio.Seq.Seq(seq_upper).reverse_complement())

    #establish common patterns for NA and HA sequences
        na_pattern = re.compile(f"CACGATAGATAAATAATAGTGCACCAT{upstream}")
        ha_pattern = re.compile(f"CCGGATTTGCATATAATGATGCACCAT{upstream}")

    #search for matches in the actucal sequence
        match_na=na_pattern.search(seq_original_string)
        match_ha=ha_pattern.search(seq_original_string)
    
    #increment counts if matches are found
        if match_na:
            match_na_count += 1
        if match_ha:
            match_ha_count += 1
    
    
    return match_na_count, match_ha_count 



from Bio.SeqRecord import SeqRecord

# made-up reads with known answers to serve as testcases 
# use SeqRecord to assign .seq to na_read. Bio.Seq.Seq is used to create the reverse complement sequence but it does not have .seq attribute. 
na_site = "CACGATAGATAAATAATAGTGCACCATAGGCGGCCGC"
ha_site = "CCGGATTTGCATATAATGATGCACCATAGGCGGCCGC"
na_read = SeqRecord(Bio.Seq.Seq("TTTT" + na_site + "ACGT").reverse_complement())
ha_read = SeqRecord(Bio.Seq.Seq("TTTT" + ha_site + "ACGT").reverse_complement())
other_read = SeqRecord(Bio.Seq.Seq("ACGTACGTACGTACGT"))

# tests: each line checks (NA count, HA count)
# ussing assert function to evaluate testcases 
assert HANA_classifier([na_read]) == (1, 0)
assert HANA_classifier([ha_read]) == (0, 1)
assert HANA_classifier([na_read, ha_read, ha_read, other_read]) == (1, 2)
print("All tests passed")

# applying HANA classifier to entire FASTQ dataset 
HANA_classifier(seqreads)
print (HANA_classifier(seqreads)[0], "reads matching to NA.", HANA_classifier(seqreads)[1], "reads matching to HA")


All tests passed
3973 reads matching to NA. 5316 reads matching to HA


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [ ]:
valid_NA_count = 0
valid_HA_count = 0 
end_of_NA = "CACGATAGATAAATAATAGTGCACCAT"
end_of_HA = "CCGGATTTGCATATAATGATGCACCAT"

#make reverse complement to obtain original sequence
for seqread in seqreads: 
    seq = seqread.seq
    seq_string = str(seq)
    seq_upper = seq_string.upper()
    seqread_original=str(Bio.Seq.Seq(seq_upper).reverse_complement())


# establish common patters for NA and HA barcodes 
    NA_barcode_pattern = re.compile(f"{end_of_NA}{upstream}(?P<barcode>[ACGTN]{{16}})")
    HA_barcode_pattern = re.compile(f"{end_of_HA}{upstream}(?P<barcode>[ACGTN]{{16}})")

#search for matches in the actual sequence 
    match_NA = NA_barcode_pattern.search(seqread_original)
    match_HA = HA_barcode_pattern.search(seqread_original)

    if match_NA:
        valid_NA_count += 1
    if match_HA:
        valid_HA_count += 1 


#subtract number of valid matches from number NA/HA reads (question 1)
invalid_NA_count = HANA_classifier(seqreads)[0] - valid_NA_count
invalid_HA_count = HANA_classifier(seqreads)[1] - valid_HA_count
 
print (f"Invalid NA barcodes: {invalid_NA_count}")
print (f"Invalid HA barcodes: {invalid_HA_count}")



Invalid NA barcodes: 61
Invalid HA barcodes: 66


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [ ]:
NA_barcode_dictionary ={}
HA_barcode_dictionary = {}

# establish common patters for NA and HA barcodes 
NA_barcode_pattern = re.compile(f"{end_of_NA}{upstream}(?P<NA_barcode>[ACGTN]{{16}})")
HA_barcode_pattern = re.compile(f"{end_of_HA}{upstream}(?P<HA_barcode>[ACGTN]{{16}})")


for seqread in seqreads: 
    seq = seqread.seq
    seq_string = str(seq)
    seq_upper = seq_string.upper()
    seqread_original=str(Bio.Seq.Seq(seq_upper).reverse_complement())

#search barcode matches in actual sequence   
    match_NA = NA_barcode_pattern.search(seqread_original)
    match_HA = HA_barcode_pattern.search(seqread_original)

#adds barcode match to respective dictionary, it will +1 for an existing barcode
    if match_NA:
        NA_barcode = match_NA.group("NA_barcode")
        NA_barcode_dictionary[NA_barcode] = NA_barcode_dictionary.get(NA_barcode, 0) + 1
    if match_HA:
        HA_barcode = match_HA.group("HA_barcode")
        HA_barcode_dictionary[HA_barcode] = HA_barcode_dictionary.get(HA_barcode, 0) + 1

max_NA_counts = 0
max_HA_counts = 0

max_NA_barcode = None
max_HA_barcode = None

#searches for most common NA and HA barcodes and their frequencies. 
for NA_barcode in NA_barcode_dictionary:
    if NA_barcode_dictionary[NA_barcode] > max_NA_counts:
        max_NA_counts = NA_barcode_dictionary[NA_barcode]
        max_NA_barcode = NA_barcode

for HA_barcode in HA_barcode_dictionary :
    if HA_barcode_dictionary[HA_barcode] > max_HA_counts:
        max_HA_counts = HA_barcode_dictionary[HA_barcode]       
        max_HA_barcode = HA_barcode


print ("Most common HA barcode is", max_HA_barcode, "with count", max_HA_counts)
print ("Most common NA barcode is", max_NA_barcode, "with count", max_NA_counts)

     
    



Most common HA barcode is CCCGACCCGACATTAA with count 155
Most common NA barcode is ACCAGTTCTCCCCGGG with count 152


In [ ]:
#References
# Lecture 8 and 9 materials.
# I used Claude (Anthropic) for writing and running the test cases for question 1, and getting suggestions for fixes on the barcode counting
#   and max-finding code in question 3.
# Christopher Choi (Staff Software Engineer at Coupang) helped with debugging and explaining how to import, 
#   set up and parse FASTQ data using BioSeqIO, so that it is usable with the re.compile functions.  